In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import ttest_ind, norm
from datetime import datetime
from scipy.stats import norm
import seaborn as sns
import random

import os

### 0. Методы добавления MDE

Допустим, ожидаемый эффект — среднее значение метрики увеличится на 10%. Рассмотрим несколько вариантов добавления эффекта:

1. Добавление константы ко всем значениям. Вычисляем среднее значение метрики, берём от него 10%, добавляем полученное число ко всем значениям экспериментальной группы.

Пример: [1, 2, 3] → [1.2, 2.2, 3.2]

2. Умножение всех значений на константу. Умножаем все значения экспериментальной группы на (1 + эффект
в долях от среднего).

Пример: [1, 2, 3] → [1.1, 2.2, 3.3]

3. Добавление константы к случайному подмножеству значений. Допустим, ожидаем, что эксперимент подействует на треть пользователей. Случайно выбираем треть значений из экспериментальной группы
и добавляем к ним константу.

Пример: [1, 2, 3] → [1.6, 2, 3] или [1, 2.6, 3], или [1, 2, 3.6]

### Task 1.

Допустим, мы переписали микросервис бэкенда, отвечающий за авторизацию. Хотим проверить, что время обработки запросов значимо не увеличится. Время обработки запроса складывается из времени авторизации и времени отработки бизнес логики. Значения времени на авторизацию для всех запросов примерно одно и то же. Время отработки бизнес логики может сильно отличаться от типа запроса. Целевая метрика — среднее время обработки запросов. Ожидаемый эффект — увеличение среднего времени выполнения запроса на 10%.

Какой способ добавления эффекта лучше подходит?

**Ответ** Добавление константы ко всем значениям

Пояснение: Время авторизации примерно равно константе, эксперимент может изменить эту константу, то есть все значения изменятся на константу

### Task 2.

Допустим, мы хотим добавить персональные рекомендации на странице корзины на нашем сайте.
Мы ожидаем, что пользователи, совершающие покупку, добавят в корзину дополнительный товар. Среди пользователей, попадающих в эксперимент, много тех, кто покупку не совершает, значение метрики для них равно нулю. Целевая метрика — средняя выручка с клиента за время эксперимента. Ожидаемый эффект — увеличение средней выручки с клиента на 3%.

Какой способ добавления эффекта лучше подходит?

**Ответ** Умножение на константу всех значений

Пояснение: Увеличатся значения клиентов, совершивших покупку. Вероятность совершить покупку не изменится

### Task 3.

Допустим, мы хотим увеличить конверсию из посещения сайта в покупку. Для этого мы сделали переход на страницу оплаты и сам процесс оплаты более простыми. Ожидаем, что из-за наших изменений увеличится доля пользователей, совершающих покупку. Целевая метрика — средняя выручка с клиента за время эксперимента. Ожидаемый эффект — увеличение средней выручки с клиента на 3%.

Какой способ добавления эффекта лучше подходит?

**Ответ** Добавление константы к некоторым значениям

Пояснение: Такое изменение похоже на совершение дополнительных покупок среди тех, кто не купил бы

### Task 4.

**Задание**

Мы рассмотрели несколько вариантов добавления эффекта. Есть ли смысл думать о способе добавления эффекта при оценке вероятности ошибки II рода или все способы дают одинаковый результат? Результаты могут быть разными. Чтобы в этом убедиться, проведём численный эксперимент.

Допустим, в наш А/В-тест попадают все пользователи, совершавшие покупки до 28 марта.

 
Целевая метрика — средняя выручка с клиента за время эксперимента. Целевую метрику считаем на неделе с 21 по 28 марта. Уровень значимости — 0.05. Критерий — тест Стьюдента. Размер групп — 1000. Ожидаемый эффект — средняя выручка увеличится на 10%.

Нужно оценить вероятности ошибок II рода для трёх вариантов добавления эффекта:

1. Добавление константы ко всем значениям;

2. Умножение на константу всех значений;

3. Добавление константы к 2.5% значений.

Для решения используйте данные из файла 2022-04-01T12_df_sales.csv.

В качестве ответа введите номера способов добавления эффекта в порядке увеличения оценки вероятности ошибки II рода. Например, если при добавлении константы ко всем значениям оценка вероятности ошибки минимальна, при добавлении константы к 2.5% значений максимальна, то ответ будет: 123.

**Решение**

In [70]:
URL_BASE = 'https://raw.githubusercontent.com/ab-courses/simulator-ab-datasets/main/2022-04-01/'

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

Необходимо учитывать, что есть клиенты (старые клиенты), которые ничего не заплатили в период эксперимента

In [71]:
sales = read_database('2022-04-01T12_df_sales.csv')
sales['date'] = pd.to_datetime(sales['date'])

In [72]:
users = sales[sales['date'] < datetime(2022, 3, 28)][['user_id']].drop_duplicates()

In [73]:
sales_hist = sales[
    (sales['date'] >= datetime(2022, 3, 21))
    & (sales['date'] < datetime(2022, 3, 28))
].groupby('user_id')[['price']].sum().reset_index()

In [74]:
grouped_sales = pd.merge(users, sales_hist, on='user_id', how='left').fillna(0)

In [75]:
mean = grouped_sales['price'].mean()
sample_size = 1000
alpha = 0.05

Вариант 1

In [79]:
# Вычисляем среднее значение метрики, берём от него 10%, добавляем полученное число ко всем значениям экспериментальной группы.
p_values = []
for _ in range(1500):
    sales_a, sales_b = np.random.choice(grouped_sales.price.astype(float), (2, sample_size,), False)
    sales_b += mean * 0.1
    _, p_val = ttest_ind(sales_a, sales_b)
    p_values.append(p_val)

errors = (np.array(p_values) > alpha).astype(int)
part_errors = np.mean(errors)
print(f'var1: part errors = {part_errors:0.4f}')

var1: part errors = 0.8000


Вариант 2

In [77]:
# Умножение всех значений на константу. Умножаем все значения экспериментальной группы на (1 + эффект в долях от среднего).
p_values = []
for _ in range(1500):
    sales_a, sales_b = np.random.choice(grouped_sales.price.astype(float), (2, sample_size,), False)
    sales_b *= 1.1
    _, p_val = ttest_ind(sales_a, sales_b)
    p_values.append(p_val)

errors = (np.array(p_values) > alpha).astype(int)
part_errors = np.mean(errors)
print(f'var1: part errors = {part_errors:0.4f}')

var1: part errors = 0.8290


Вариант 3

In [80]:
# Добавление константы к случайному подмножеству значений. Допустим, ожидаем, что эксперимент подействует на треть пользователей. 
#Случайно выбираем треть значений из экспериментальной группы и добавляем к ним константу.
p_values = []
for _ in range(1500):
    sales_a, sales_b = np.random.choice(grouped_sales.price.astype(float), (2, sample_size,), False)
    indexes = np.random.choice(np.arange(sample_size), int(sample_size * 0.025), False)
    add_value = 0.1 * mean * sample_size / len(indexes)
    mask = np.zeros(sample_size)
    mask[indexes] += 1
    sales_b += mask * add_value
    
    _, p_val = ttest_ind(sales_a, sales_b)
    p_values.append(p_val)

errors = (np.array(p_values) > alpha).astype(int)
part_errors = np.mean(errors)
print(f'var1: part errors = {part_errors:0.4f}')

var1: part errors = 0.7980


Правильное решение

In [68]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats

URL_BASE = 'https://raw.githubusercontent.com/ab-courses/simulator-ab-datasets/main/2022-04-01/'

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

df_sales = read_database('2022-04-01T12_df_sales.csv')
df_sales['date'] = pd.to_datetime(df_sales['date'])

begin_date = datetime(2022, 3, 21)
end_date = datetime(2022, 3, 28)
df_users = df_sales[df_sales['date'] < end_date][['user_id']].drop_duplicates()
df_metrics = (
    df_sales
    [(df_sales['date'] >= begin_date) & (df_sales['date'] < end_date)]
    .groupby('user_id')[['price']].sum()
    .reset_index()  
)
df = pd.merge(df_users, df_metrics, on='user_id', how='left').fillna(0)

alpha = 0.05
sample_size = 1000
effect = 0.1

pvalues = {'one': [], 'two': [], 'three': []}
values = df['price'].values
mean_ = values.mean()

for _ in range(30000):
    # выбираем случайные группы
    a, b = np.random.choice(values, (2, sample_size,), False)
    # добавляем эффект тремя способами
    b_one = b + mean_ * effect
    b_two = b * (1 + effect)
    indexes = np.random.choice(np.arange(sample_size), int(sample_size * 0.025), False)
    add_value = effect * mean_ * sample_size / len(indexes)
    mask = np.zeros(sample_size)
    mask[indexes] += 1
    b_three = b + mask * add_value
    # считаем и сохраняем p-value
    for b_, key in ((b_one, 'one',), (b_two, 'two',), (b_three, 'three',),):
        pvalues[key].append(stats.ttest_ind(a, b_).pvalue)

# считаем точечные оценки вероятностей ошибки II рода
for key, v in pvalues.items():
    errors = (np.array(v) > alpha).astype(int)
    part_errors = np.mean(errors)
    print(f'{key}: part errors = {part_errors:0.4f}')

# проверим, что отличия статистически значимые
print(stats.ttest_ind(pvalues['one'], pvalues['three']).pvalue)
print(stats.ttest_ind(pvalues['two'], pvalues['three']).pvalue)

one: part errors = 0.8158
two: part errors = 0.8303
three: part errors = 0.8262
0.006457480850549465
0.022136766550999613


### Task 5.

**Задание**

Сегодня нужно будет реализовать код для оценки вероятностей ошибок I и II рода.

Напишите функцию estimate_errors.

Шаблон решения

In [11]:
import numpy as np
import pandas as pd
from scipy import stats


def estimate_errors(group_generator, effect_add_type, effect, alpha):
    """Оцениваем вероятности ошибок I и II рода.

    :param group_generator: генератор значений метрик для двух групп.
    :param effect_add_type (str): способ добавления эффекта для группы B.
        - 'all_const' - увеличить всем значениям в группе B на константу (b_metric_values.mean() * effect / 100).
        - 'all_percent' - увеличить всем значениям в группе B в (1 + effect / 100) раз.
    :param effect (float): размер эффекта в процентах.
        Пример, effect=3 означает, что ожидаем увеличение среднего на 3%.
    :param alpha (float): уровень значимости.
    :return pvalues_aa (list[float]), pvalues_ab (list[float]), first_type_error (float), second_type_error (float):
        - pvalues_aa, pvalues_ab - списки со значениями pvalue
        - first_type_error, second_type_error - оценки вероятностей ошибок I и II рода.
    """
    # YOUR_CODE_HERE

Обратите внимание, что на вход функции подаются не исходные значения метрики, а генератор, который выдаёт уже сэмплированные данные. Это позволит нам детерминировано протестировать правильноcть решения. Самостоятельно сэмлировать данные внутри функции estimate_errors не нужно.

Пример реализации генератора group_generator

In [15]:
def create_group_generator(metrics, sample_size, n_iter):
    """Генератор случайных групп.

    :param metrics (pd.DataFame): таблица с метриками, columns=['user_id', 'metric'].
    :param sample_size (int): размер групп (количество пользователей в группе).
    :param n_iter (int): количество итераций генерирования случайных групп.
    :return (np.array, np.array): два массива со значениями метрик в группах.
    """
    user_ids = metrics['user_id'].unique()
    for _ in range(n_iter):
        a_user_ids, b_user_ids = np.random.choice(user_ids, (2, sample_size), False)
        a_metric_values = metrics.loc[metrics['user_id'].isin(a_user_ids), 'metric'].values
        b_metric_values = metrics.loc[metrics['user_id'].isin(b_user_ids), 'metric'].values
        yield a_metric_values, b_metric_values

metrics = pd.DataFrame({'user_id': [1, 2, 3, 4], 'metric': [5, 6, 8, 9.1] })
sample_size = 2
n_iter = 3
group_generator = create_group_generator(metrics, sample_size, n_iter)

for metrics_a_group, metrics_b_group in group_generator:
    print(metrics_a_group, metrics_b_group)
# >>> [8.  9.1] [5. 6.]
# >>> [5.  9.1] [6. 8.]
# >>> [5. 6.] [8.  9.1]

[5.  9.1] [6. 8.]
[5. 8.] [6.  9.1]
[5. 6.] [8.  9.1]


**Решение**

In [18]:
import numpy as np
import pandas as pd
from scipy import stats


def estimate_errors(group_generator, effect_add_type, effect, alpha):
    """Оцениваем вероятности ошибок I и II рода.

    :param group_generator: генератор значений метрик для двух групп.
    :param effect_add_type (str): способ добавления эффекта для группы B.
        - 'all_const' - увеличить всем значениям в группе B на константу (b_metric_values.mean() * effect / 100).
        - 'all_percent' - увеличить всем значениям в группе B в (1 + effect / 100) раз.
    :param effect (float): размер эффекта в процентах.
        Пример, effect=3 означает, что ожидаем увеличение среднего на 3%.
    :param alpha (float): уровень значимости.
    :return pvalues_aa (list[float]), pvalues_ab (list[float]), first_type_error (float), second_type_error (float):
        - pvalues_aa, pvalues_ab - списки со значениями pvalue
        - first_type_error, second_type_error - оценки вероятностей ошибок I и II рода.
    """
    pvalues_aa = []
    pvalues_ab = []
    aa_errors = 0
    ab_errors = 0
    aa_tests_count = 0
    ab_tests_count = 0

    for metrics_a_group, metrics_b_group in group_generator:
        #A/A test
        t_stat_aa, p_value_aa = stats.ttest_ind(metrics_a_group, metrics_b_group)
        pvalues_aa.append(p_value_aa)
        if p_value_aa < alpha:
            aa_errors += 1
        aa_tests_count += 1
        
        #Add effect to metric b
        if effect_add_type == 'all_const':
            effect_value = metrics_b_group.mean() * effect / 100
            metrics_b_group += effect_value
        elif effect_add_type == 'all_percent':
            metrics_b_group *= 1 + effect / 100

        # A/B test
        t_stat_ab, p_value_ab = stats.ttest_ind(metrics_a_group, metrics_b_group)
        pvalues_ab.append(p_value_ab)
        if p_value_ab >= alpha:
            ab_errors += 1
        ab_tests_count += 1

        #Type I error: false positive
        typeI_error = aa_errors / aa_tests_count

        #Type II error: false negative
        typeII_error = ab_errors / ab_tests_count

    return pvalues_aa, pvalues_ab, typeI_error, typeII_error

Применение

In [19]:
sample_size, n_iter, effect, alpha = 100, 10, 6, 0.05

group_generator = (
    (np.arange(sample_size, dtype=float), np.arange(sample_size, dtype=float) + x,)
    for x in range(n_iter)
)
effect_add_type = 'all_const'
pvalues_aa, pvalues_ab, first_type_error, second_type_error = estimate_errors(
    group_generator, effect_add_type, effect, alpha
)
# pvalues_aa = [1.0, 0.808, 0.626, 0.466, 0.331, 0.224, 0.145, 0.09, 0.053, 0.029]
# pvalues_ab = [0.47, 0.327, 0.216, 0.135, 0.08, 0.045, 0.024, 0.012, 0.006, 0.003]
# first_type_error = 0.1
# second_type_error = 0.5

In [20]:
pvalues_aa

[np.float64(1.0),
 np.float64(0.8076897014876132),
 np.float64(0.626466928391483),
 np.float64(0.46552142375730854),
 np.float64(0.33078290948802747),
 np.float64(0.22442071600176872),
 np.float64(0.14521704796052834),
 np.float64(0.08955122234817349),
 np.float64(0.05260398295958351),
 np.float64(0.02942842378712022)]